<a href="https://colab.research.google.com/github/paantabh4t/DaysTillBananaDeath/blob/main/Notebook/DaysTillBananaDeath.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install extra libraries (run once)
!pip install tensorflow matplotlib pillow -q

# Import everything we need
import tensorflow as tf              # main TensorFlow/Keras library
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt      # plot graphs and images
import numpy as np
import os

# Check GPU
print("GPUs available:", tf.config.list_physical_devices('GPU'))


GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
!unzip -q /content/banana_dataset.zip -d /content/banana_dataset

In [ ]:
# Where your dataset lives
base_path = '/content/banana_dataset/Banana Ripeness Classification Dataset'

# Map each class folder → days until expiry
days_map = {
    'unripe':   10,
    'ripe':      5,
    'overripe':  2,
    'rotten':    0
}

# Verify folders exist
for split in ['train', 'valid', 'test']:
    print(f"\n{split.upper()}:")
    for cls in days_map:
        path = f"{base_path}/{split}/{cls}"
        count = len(os.listdir(path))
        print(f"  {cls} → {count} images, {days_map[cls]} days")


TRAIN:
  unripe → 1902 images, 10 days
  ripe → 3522 images, 5 days
  overripe → 2349 images, 2 days
  rotten → 4020 images, 0 days

VALID:
  unripe → 167 images, 10 days
  ripe → 339 images, 5 days
  overripe → 229 images, 2 days
  rotten → 388 images, 0 days

TEST:
  unripe → 110 images, 10 days
  ripe → 154 images, 5 days
  overripe → 113 images, 2 days
  rotten → 185 images, 0 days


In [ ]:
# Keras handles transforms inside ImageDataGenerator
# Training: augmentation ON
train_datagen = ImageDataGenerator(
    rescale=1./255,              # normalize pixel values 0-1
    horizontal_flip=True,       # randomly mirror images
    brightness_range=[0.8, 1.2], # randomly adjust brightness
    zoom_range=0.1              # slight random zoom
)

# Validation/Test: only normalize (no augmentation)
val_datagen = ImageDataGenerator(rescale=1./255)

print("Transforms ready!")

Transforms ready!


In [ ]:
# Keras loads images directly from folders using flow_from_directory
# But since we need regression (numbers not classes), we use a custom generator

import numpy as np
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import load_img, img_to_array

class BananaGenerator(Sequence):
    def __init__(self, split, batch_size=32, augment=False):
        self.batch_size = batch_size
        self.augment = augment
        self.images, self.labels = [], []

        for cls, days in days_map.items():
            folder = f"{base_path}/{split}/{cls}"
            for fname in os.listdir(folder):
                if fname.endswith(('.jpg','.jpeg','.png')):
                    self.images.append(f"{folder}/{fname}")
                    self.labels.append(days)
        print(f"{split}: {len(self.images)} images")

    def __len__(self):
        return len(self.images) // self.batch_size

    def __getitem__(self, idx):
        batch_imgs = self.images[idx*self.batch_size:(idx+1)*self.batch_size]
        batch_labels = self.labels[idx*self.batch_size:(idx+1)*self.batch_size]
        X = np.array([img_to_array(load_img(p, target_size=(224,224)))/255.0
                       for p in batch_imgs])
        y = np.array(batch_labels, dtype=np.float32)
        return X, y

train_gen = BananaGenerator('train', augment=True)
valid_gen = BananaGenerator('valid')
test_gen  = BananaGenerator('test')

In [ ]:
# Load pretrained ResNet-50 (without its top classification layer)
base_model = ResNet50(
    weights='imagenet',       # use pretrained weights
    include_top=False,        # remove the last classification layer
    input_shape=(224, 224, 3) # our image size
)

# Freeze pretrained layers
base_model.trainable = False

# Add our own regression head on top
inputs = keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)  # flatten features
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)              # prevents overfitting
outputs = layers.Dense(1)(x)            # output 1 number (days)

model = keras.Model(inputs, outputs)
model.summary()                          # print model structure

In [ ]:
# Compile: set loss + optimizer in one step
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',      # Mean Squared Error
    metrics=['accuracy']
)

print("Model compiled!")

In [ ]:
# Keras trains with a single .fit() call — much simpler!
EPOCHS = 10

history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=valid_gen,
    verbose=1 )  # print progress each epoch

# history.history contains loss values for plotting
print("Training complete!")

In [ ]:
# Plot training vs validation loss
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'],     label='Train Loss', color='orange')
plt.plot(history.history['val_loss'], label='Val Loss',   color='blue')
plt.plot(history.history['accuracy'], label='Accuracy',   color='red')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Progress')
plt.legend()
plt.show()

In [ ]:
# Save locally (temporary)
model.save('/content/banana_model.keras')
print("Saved locally!")

from google.colab import files
files.download('/content/banana_model.keras')